In [ ]:
import numpy as np
import math

In [ ]:
#### Reference_data ####
temp = np.array([300,350,400,450,500,550,600,650,700,750,800,900,1000,1100])

c_sc = np.array([1.44,4.51,7.94,11.65,15.58,19.69,23.94,28.31,32.77,37.31,41.92,51.29,60.82,70.48])*10**-4
a_sc = np.array([1.29,3.98,7.01,10.32,13.83,17.52,21.34,25.28,29.30,33.40,37.57,46.05,54.69,63.45])*10**-4
v_pol = np.array([1.51,4.43,7.67,11.17,14.85,18.7,22.67,26.74,30.89,35.12,39.40,48.11,56.97,65.95])*10**-4  


#### expansion equations (second degree poly) for cell parameters a and c, and volume ####
a_fit = np.polyfit(temp,a_sc,2)  # used in the second part of the script
c_fit = np.polyfit(temp,c_sc,2)  # used in the second part of the script
v_fit = np.polyfit(temp,v_pol,2) 


####temperature calculation function####
def solv_eq_T(array,par_std,par_T):
    a , b , c = array
    dap = (par_T-par_std)/par_std    
    c_cor = c - dap                  #the second degree poly is not equal to 0 but to the da/a (here dap)
    delta = b**2-4*a*c_cor
    if delta<0:
        return(f"Negative Delta!")
    temp_1=(-b+math.sqrt(delta))/(2*a) - 273.15  

    return(temp_1)


####cell parameter calculation function####
def solv_eq_par(array, T, par_std):
    a , b , c = array
    par_T = par_std*(1 + a*T**2 + b*T + c)
    
    return(f'par_T: {par_T} at {T}°C')

In [ ]:
################ SECOND PART: MULTIPLE TEMPERATURES ########################################
############ you can use the functions alone or apply them to a file with a set of temperatures ###########

#### experimental data: output file from topas ---> Al2O3_seq.inp ####
fname = r'C:\Users\giulia01a\Documents\PhD\Experiments\CH-7630\Al2O3\pedavero\results_test1.txt'  
index , temp_Al_C , temp_Al_K , a_al , c_al , vol_al = np.loadtxt( fname , skiprows = 1).T   #check topas output file format from macro in local.inc


####calculation of T from different coefficients####
temp_calc_a = solv_eq_T(a_fit,4.75994, a_al)
temp_calc_c = solv_eq_T(c_fit,12.99429, c_al)
temp_calc_v = solv_eq_T(v_fit,254.96786**(1/3), vol_al**(1/3))

t_aver = np.nanmean([temp_calc_a , temp_calc_c , temp_calc_v], axis =0)    #average (or choose the most convenient)


#### output generation 1 and 2 ####
all_temps = np.savetxt('out_test_watchman.txt' , np.c_[temp_Al_C , temp_calc_a , temp_calc_c , temp_calc_v , t_aver] , header ='T_C_exp T_calc_a T_calc_c T_calc_pV T_aver ' , fmt = '%.2f')
#all_temps = np.savetxt('out_test_watchman.txt' , np.c_[temp_Al_C , t_aver, a_al , c_al , vol_al] , header = 'T_C_exp T_aver par_a par_c Vol' , fmt = '%.2f')
